<a href="https://colab.research.google.com/github/Aradhyagodambe/quire/blob/main/quire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -qU pymupdf4llm llama-index transformers accelerate bitsandbytes sentence-transformers llama-index-embeddings-huggingface llama-index-llms-huggingface

In [ ]:
# !pip install -qU llama-index-postprocessor-sbert-rerank

In [ ]:
import torch
from transformers import BitsAndBytesConfig
from llama_index.core import Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_compute_dtype= torch.float16,
    bnb_4bit_quant_type= "nf4"
)

In [ ]:
llm = HuggingFaceLLM(
    model_name = "HuggingFaceH4/zephyr-7b-beta",
    tokenizer_name = "HuggingFaceH4/zephyr-7b-beta",
    context_window = 8192,
    max_new_tokens = 512,
    model_kwargs = {"quantization_config" : bnb_config},
    generate_kwargs = {"temperature" : 0.1, "do_sample" : True},
    device_map = "auto"
)

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(model_name = "sentence-transformers/all-MiniLM-L6-v2")
Settings.chunk_size = 512
Settings.chunk_overlap = 50

In [ ]:
import pymupdf4llm
from llama_index.core import Document, VectorStoreIndex
from google.colab import files

In [ ]:
uploaded = files.upload()

documents = []

Saving research ppr 6.pdf to research ppr 6 (1).pdf
Saving research ppr 5.pdf to research ppr 5 (1).pdf
Saving research ppr 4.pdf to research ppr 4 (1).pdf
Saving research ppr 3.pdf to research ppr 3 (1).pdf
Saving research ppr 2.pdf to research ppr 2 (1).pdf
Saving research ppr.pdf to research ppr (1).pdf


In [ ]:
for pdf_path in uploaded.keys():
  print(pdf_path)

  page_data = pymupdf4llm.to_markdown(pdf_path, page_chunks = True)

  for page in page_data:
    current_page_num = page.get("metadata", {}).get("page",0) +1

    doc = Document(
        text = page["text"],
        metadata = {
            "filename" : pdf_path,
            "page_number" : current_page_num
        }
    )
    documents.append(doc)

research ppr 6 (1).pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=6/7.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=11/12.
OCR on page.number=13/14.
research ppr 5 (1).pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.
OCR on page.number=5/6.
OCR on page.number=7/8.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=10/11.
OCR on page.number=13/14.
research ppr 4 (1).pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=4/5.
OCR on page.number=14/15.
OCR on page.number=18/19.
OCR on page.number=19/20.
 page.number=9/10.
OCR on page.number=10/11.
research ppr 3 (1).pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=6/7.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.nu

In [ ]:
print(f"Building Vector Index {len(documents)} ")
index = VectorStoreIndex.from_documents(documents)

Building Vector Index 103 


In [ ]:
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.vector_stores import MetadataFilter, ExactMatchFilter
from llama_index.postprocessor.sbert_rerank import SentenceTransformerRerank

In [ ]:
reranker = SentenceTransformerRerank(
    model = "cross-encoder/ms-marco-miniLM-L-6-v2",
    top_n = 3
)

shared_memory = ChatMemoryBuffer.from_defaults(token_limit = 1500)

In [ ]:
from llama_index.core import PromptTemplate

# 1. Create a simpler, highly direct prompt for the 7B model
custom_condense_prompt = PromptTemplate(
    "Given the following conversation history and a follow-up question, rewrite the follow-up question into a standalone query that contains all necessary context.\n\n"
    "Chat History:\n"
    "{chat_history}\n\n"
    "Follow Up Input: {question}\n\n"
    "Standalone query:"
)

In [ ]:
def get_chat_engine(target_files=None):

    filters = None

    if target_files:

        if isinstance(target_files, str):
            target_files = [target_files]

        filters = MetadataFilters(
            filters=[
                MetadataFilter(
                    key="filename",
                    value=target_files,
                    operator=FilterOperator.IN
                )
            ]
        )

    return index.as_chat_engine(
        chat_mode="condense_plus_context",
        memory=shared_memory,
        filters=filters,
        similarity_top_k=8,
        node_postprocessors=[reranker],
        condense_prompt = custom_condense_prompt #<----- Remove if block above is to be deleted
    )

In [ ]:
global_engine = get_chat_engine(target_files = None)

query_1 = "Summarize the abstracts and conclusions found in these documents."
print(f"\nQuestion :  {query_1}")
response_1 = global_engine.chat(query_1)
print(f"\nAnswer:\n {response_1.response}\n")


targeted_engine = get_chat_engine(target_files="research ppr 2.pdf")

query_2 = "How does this specific paper address those themes?"
print(f"\nQuestion : {query_2}")
response_2 = targeted_engine.chat(query_2)
print(f"\nAnswer:\n {response_2.response}\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Question :  Summarize the abstracts and conclusions found in these documents.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Answer:
 Document 1: "Research PPR 4 (1)"

Abstract: The paper proposes a model for anomaly detection in financial transactions using machine learning techniques. The model is validated through simulation results, demonstrating high true positive and true negative rates for identifying positive and negative classes.

Conclusion: The proposed model is validated through simulation results, showing high true positive and true negative rates for identifying positive and negative classes.

Document 2: "Research PPR 2 (1)"

Abstract: This paper defines the features of the financial transaction dataset used in the study, including step, type, amount, identification credentials, account balances, and fraud flags. The dataset is imbalanced, with only 0.0012% of instances being fraudulent, and fraud cases occur only in transfer and cash-out transactions.

Conclusion: The paper defines the features of the financial transaction dataset used in the study, including step, type, amount, identificati

In [ ]:
print("\n--- Source Citations ---")

for i, node in enumerate(response_2.source_nodes, 1):
  filename = node.metadata.git("filename", "Unknown")
  page_num = node.metadata.get("page_number", "Unknown")
  score = round(node.score, 3) if node.score else "N/A"
  print(f"Source {i} : {filename} (Page {page_num}) - Ranker Score: {score}")


--- Source Citations ---
